# Evaluate on a Colab A100

This notebook is a batch job. It serves the baseline and candidate with vLLM on this GPU, runs the selected suite through `gate run`, stops both servers, and copies `runs/<run_id>/` to Google Drive. Scoring and the gate decision run locally on the saved run.

Requirements:
- A Colab A100 runtime.
- A Colab secret named `HF_TOKEN` with access to gated models such as Llama. Accept the model license on Hugging Face first.

Compute units are billed while the runtime is connected. Run all cells in order (Drive sign-in happens early, before the GPU servers start), then disconnect as the last cell says.

In [ ]:
REPO_URL = "https://github.com/abhishek-paul-d/LLM_GATE.git"
BRANCH = "main"
BASELINE = "llama-3.1-8b-instruct-fp8"
CANDIDATE = "qwen3-8b-fp8"
SUITE = "suites/starter-v1"
SPLIT = "release"
PROMPT = "prompts/triage-v1.yaml"
PORTS = {"baseline": 8001, "candidate": 8002}
GPU_MEMORY_UTILIZATION = 0.45
DRIVE_DIR = "/content/drive/MyDrive/llm-release-gate/runs"

In [ ]:
!nvidia-smi

In [ ]:
import os

if not os.path.isdir("/content/LLM_GATE"):
    !git clone --branch {BRANCH} {REPO_URL} /content/LLM_GATE
%cd /content/LLM_GATE
!pip install -q -e . vllm

In [ ]:
import os

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
# Mount Drive now, before the GPU servers start, so the sign-in prompt never holds up a billed session later.
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import torch

from release_gate import registry

baseline_spec = registry.load_model_spec(BASELINE)
candidate_spec = registry.load_model_spec(CANDIDATE)
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
specs = [baseline_spec, candidate_spec]
# Refuse before any server starts: the manifest records the revision as given, so a branch
# such as "main" would not identify the weights that were served.
unpinned = [spec.name for spec in specs if not spec.revision_pinned]
if unpinned:
    raise RuntimeError(f"Pin model.revision to a 40-char commit SHA in models/<name>.yaml first: {unpinned}")
if not registry.fits_concurrently(specs, gpu_gb):
    raise RuntimeError(
        "These models do not fit concurrently. Sequential mode is not implemented yet; use smaller or quantized specs."
    )
print("Revisions:", {spec.name: spec.model.revision[:12] for spec in specs})
print(f"Combined model weights: {sum(spec.serving.weights_gb for spec in specs):.1f} GB; GPU memory: {gpu_gb:.1f} GB")

In [ ]:
import importlib.metadata
import json
import subprocess
from pathlib import Path

nvidia_smi = subprocess.check_output(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,driver_version",
        "--format=csv,noheader",
    ],
    text=True,
).strip()
gpu_name, gpu_memory_total, driver_version = [part.strip() for part in nvidia_smi.split(",", 2)]
environment = {
    "gpu_name": gpu_name,
    "gpu_memory_total": gpu_memory_total,
    "driver_version": driver_version,
    "cuda_version": str(torch.version.cuda or "unknown"),
    "torch_version": str(torch.__version__),
    "vllm_version": importlib.metadata.version("vllm"),
    "execution_mode": "concurrent",
    "gpu_memory_utilization": str(GPU_MEMORY_UTILIZATION),
}
Path("/content/env.json").write_text(json.dumps(environment, indent=2) + "\n", encoding="utf-8")
environment

In [ ]:
import time

import httpx

Path("/content/logs").mkdir(parents=True, exist_ok=True)
servers = {}
log_files = {}


def wait_healthy(role, port, timeout_s=900):
    deadline = time.monotonic() + timeout_s
    url = f"http://127.0.0.1:{port}/v1/models"
    while time.monotonic() < deadline:
        if servers[role].poll() is not None:
            break
        try:
            if httpx.get(url, timeout=5).status_code == 200:
                print(f"{role} server is healthy")
                return
        except httpx.HTTPError:
            pass
        time.sleep(5)
    log_path = Path(f"/content/logs/{role}.log")
    tail = log_path.read_text(errors="replace").splitlines()[-50:] if log_path.exists() else []
    print(f"{role} server log tail:\n" + "\n".join(tail))
    raise RuntimeError(f"{role} server exited or did not become healthy within {timeout_s} seconds")


def start_server(role, spec):
    log_file = open(f"/content/logs/{role}.log", "w")
    log_files[role] = log_file
    command = spec.vllm_serve_args(PORTS[role], GPU_MEMORY_UTILIZATION)
    servers[role] = subprocess.Popen(command, stdout=log_file, stderr=subprocess.STDOUT, env=os.environ.copy())


def stop_servers():
    for process in servers.values():
        if process.poll() is None:
            process.terminate()
    for process in servers.values():
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
    for log_file in log_files.values():
        log_file.close()
    servers.clear()
    log_files.clear()
    print("vLLM servers stopped.")

In [ ]:
# Start the baseline, wait until it is healthy, then the candidate, so the two servers do not race
# for GPU memory. The finally block stops both servers even if a health check or the run fails.
from datetime import UTC, datetime

RUN_ID = f"{datetime.now(UTC):%Y%m%dT%H%M%SZ}-{BASELINE}-vs-{CANDIDATE}"
RUN_DIR = Path("runs") / RUN_ID
try:
    start_server("baseline", baseline_spec)
    wait_healthy("baseline", PORTS["baseline"])
    start_server("candidate", candidate_spec)
    wait_healthy("candidate", PORTS["candidate"])
    subprocess.run(
        [
            "gate",
            "run",
            "--suite",
            SUITE,
            "--split",
            SPLIT,
            "--prompt",
            PROMPT,
            "--baseline",
            BASELINE,
            "--candidate",
            CANDIDATE,
            "--baseline-url",
            f"http://127.0.0.1:{PORTS['baseline']}/v1",
            "--candidate-url",
            f"http://127.0.0.1:{PORTS['candidate']}/v1",
            "--env-file",
            "/content/env.json",
            "--run-id",
            RUN_ID,
        ],
        check=True,
    )
finally:
    stop_servers()
print("Saved run:", RUN_DIR)

In [ ]:
import shutil

destination = Path(DRIVE_DIR) / RUN_ID
destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(RUN_DIR, destination)
print("Copied run to", destination)

## Disconnect now

Use **Runtime > Disconnect and delete runtime** now to stop compute-unit billing. The run directory is on Google Drive and can be scored and gated locally.